<a href="https://colab.research.google.com/github/keathana2007/DEEP-LEARNING/blob/main/dl_skill_8_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import time
import numpy as np

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.14.1+cu130
CUDA Available: True


In [ ]:
if torch.cuda.is_available():

    print("GPU is available!")

    print("GPU Name:",
          torch.cuda.get_device_name(0))

    print("CUDA Version:",
          torch.version.cuda)

else:
    print("GPU is not available. Using CPU.")

GPU is available!
GPU Name: Tesla T4
CUDA Version: 13.0


In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.model = nn.Sequential(

            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),

            nn.Flatten(),

            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)


model = SimpleCNN()

print(model)

SimpleCNN(
  (model): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): AdaptiveAvgPool2d(output_size=(1, 1))
    (8): Flatten(start_dim=1, end_dim=-1)
    (9): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [ ]:
# Create 10 sample images

images = torch.randn(10, 3, 128, 128)

print("Image Shape:", images.shape)

print("Total Images:", len(images))

Image Shape: torch.Size([10, 3, 128, 128])
Total Images: 10


In [ ]:
# Move model to CPU

cpu_model = model.to("cpu")

cpu_model.eval()

# Run inference

start_time = time.time()

with torch.no_grad():

    cpu_output = cpu_model(images)

end_time = time.time()

cpu_time = end_time - start_time

print("CPU Inference Completed!")

print("Output Shape:", cpu_output.shape)

print("CPU Inference Time:",
      round(cpu_time, 4), "seconds")

CPU Inference Completed!
Output Shape: torch.Size([10, 10])
CPU Inference Time: 0.2808 seconds


In [ ]:
# Move model to CPU

cpu_model = model.to("cpu")

cpu_model.eval()

# Run inference

start_time = time.time()

with torch.no_grad():

    cpu_output = cpu_model(images)

end_time = time.time()

cpu_time = end_time - start_time

print("CPU Inference Completed!")

print("Output Shape:", cpu_output.shape)

print("CPU Inference Time:",
      round(cpu_time, 4), "seconds")

CPU Inference Completed!
Output Shape: torch.Size([10, 10])
CPU Inference Time: 0.1202 seconds


In [ ]:

print("----- PERFORMANCE COMPARISON -----")

print("CPU Time:",
      round(cpu_time, 4), "seconds")

if torch.cuda.is_available():

    if 'gpu_time' in globals():

        print("GPU Time:",
              round(gpu_time, 4), "seconds")

        if gpu_time > 0:

            speedup = cpu_time / gpu_time

            print("GPU Speedup:",
                  round(speedup, 2), "times")

    else:
        print("GPU time not found!")
        print("Please run Step 6 again.")

else:
    print("GPU is not available!")


----- PERFORMANCE COMPARISON -----
CPU Time: 0.1202 seconds
GPU time not found!
Please run Step 6 again.


In [ ]:
if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated(0) / (1024 ** 2)

    reserved = torch.cuda.memory_reserved(0) / (1024 ** 2)

    print("GPU Memory Analysis")

    print("Allocated Memory:",
          round(allocated, 2), "MB")

    print("Reserved Memory:",
          round(reserved, 2), "MB")

else:
    print("GPU is not available")

GPU Memory Analysis
Allocated Memory: 0.0 MB
Reserved Memory: 0.0 MB


In [ ]:

import torch
import time

if torch.cuda.is_available():

    # Create GPU model
    gpu_model = SimpleCNN().to("cuda")
    gpu_model.eval()

    # Transfer images to GPU
    gpu_images = images.to("cuda")

    # Warm-up
    for i in range(10):
        with torch.no_grad():
            output = gpu_model(gpu_images)

    torch.cuda.synchronize()

    # Measure data transfer time
    start = time.time()

    for i in range(30):
        gpu_images = images.to("cuda")

    torch.cuda.synchronize()

    transfer_time = (time.time() - start) / 30

    # Measure inference time
    start = time.time()

    for i in range(30):
        with torch.no_grad():
            output = gpu_model(gpu_images)

    torch.cuda.synchronize()

    inference_time = (time.time() - start) / 30

    # Total time
    total_time = transfer_time + inference_time

    print("----- BOTTLENECK ANALYSIS -----")

    print("Data Transfer Time:",
          round(transfer_time * 1000, 3), "ms")

    print("Inference Time:",
          round(inference_time * 1000, 3), "ms")

    print("Total Time:",
          round(total_time * 1000, 3), "ms")

    if transfer_time > inference_time:
        print("Possible Bottleneck: Data Transfer")
    else:
        print("Possible Bottleneck: Model Execution")

else:
    print("GPU is not available!")



----- BOTTLENECK ANALYSIS -----
Data Transfer Time: 0.518 ms
Inference Time: 2.538 ms
Total Time: 3.056 ms
Possible Bottleneck: Model Execution
